# Part 1


## Introduction


In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "rl_suite").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break
import gymnasium as gym
import pprint  # useful for printing nested items

import rl_suite as rl
import rl_suite.algorithms as algs

utils = rl.RLToolbox.utils


In [2]:
BASE_ENV_ID, SEED = "CartPole-v1", 10
NUM_TIMESTEPS_GOAL = 10000
FOUR_DIM_DISCRETIZATION = {
    "num_bins": 10,
    "intervals": ((-2.4, 2.4), (-2.5, 2.5), (-0.2095, 0.2095), (-3.5, 3.5)),
}

TWO_DIM_DISCRETIZATION = {
    "num_bins": 100,
    "feature_indices": (0, 2),
    "intervals": ((-2.4, 2.4), (-0.2095, 0.2095)),
}

FOUR_DIM_ENV_ID = utils.register_discretized_env(
    BASE_ENV_ID,
    id="rl_suite/CartPoleFourDimDiscrete-v0",
    discretization=FOUR_DIM_DISCRETIZATION,
    env_kwargs={"max_episode_steps": NUM_TIMESTEPS_GOAL},
    force=True,
)
TWO_DIM_ENV_ID = utils.register_discretized_env(
    BASE_ENV_ID,
    id="rl_suite/CartPoleTwoDimDiscrete-v0",
    discretization=TWO_DIM_DISCRETIZATION,
    env_kwargs={"max_episode_steps": NUM_TIMESTEPS_GOAL},
    force=True,
)

In [3]:
# ``print_discrete_space`` is accessed through ``rl_suite.RLToolbox.utils``.


## Approach

It is clear that the duration of this control algorithm depends directly on how we discretize the continuous feedback; so our execution loop will be a function of the fineness of discretization. 

The algorithm for off-policy MC control algorithm involves using an infinite amount of episodes but this is obviously infeasible. We need to generate enough episodes that the agent can actually solve/optimize the problem (the target policy must converge to a deterministic optimum). The instructions say the goal is for the solver to balance it for 10,000 steps. Then our goal is to keep generating episodes until we finally reach an episode that is 10,000 timesteps long, however this is impractical so we implemented a maximum number of iterations of 100,000. Since the behaviour policy used to generate the episodes is arbitrarily soft and independent of the target policy being learned (except for coverage), we cannot use these episodes to test whether our algorithm has converged. So for every 1000 iterations, we generate an episode using the target policy as a test. Therefore, the agent has 100 "test attempts" to balance the cart for 10,000 timesteps, and 100,000 episodes to learn from. If the agent still does not accomplish this, it has failed.

We implemented three modified versions of the Off-Policy Control algorithm; each agent failed. We will present each implementation, discuss the changes, and afterward discuss the limitations of this approach. The "logs" for these algorithms can be found in Appendices B, C, and D.

Here is the main loop used to execute the environment (generate an episode) for agents 1 and 2:

In [4]:
# Environment execution now uses registered discretized Gymnasium envs.

The shared off-policy Monte Carlo implementation lives in `rl_suite.algorithms`; this notebook registers discretized CartPole variants and then creates them through `gym.make`, so the algorithms receive a normal Gymnasium environment with `MultiDiscrete` observations.

In [5]:
# Off-policy MC control is available as ``algs.mc.off_policy(env)``.


## Experiment 1

This first experiment uses `OffPolicyMCAgent` with a four-dimensional CartPole discretization. The observed feedback includes cart position, cart velocity, angle (in radians), and angular velocity. Since it is infeasible to have an infinite state space for MC control, we manually limit the range of the cart velocity and the angular velocity. We chose the smallest range that included every single observed value from initial testing; these were $(-2.5,2.5)$ and $(-3.5,3.5)$ for cart velocity and angular velocity, respectively.

As mentioned earlier, discretization is necessary, and we decided to settle on 10 "bins" for each variable (so each variable has 10 possible values). Thus, the size of our state space is $|S| = 10^4 = 10000$, and since there are only two possible actions (push left or push right), our state-action is of size $|S \times A| = 10000 \times 2 = 20000$. For each state variables we ensure that there is an equal number of bins representing negative values as positive values (the edge case of 0 is irrelevant). Thus the number of bins are forced to be even.

The behaviour policy simply chooses between action 0 and 1 with equal probability for any state. Thus it is a soft policy and the assumption of coverage is still valid (any state-action pair possible under the target policy is possible under behaviour).

In [6]:
# Experiment 1: four-dimensional CartPole discretization with a uniform behaviour policy.


In [7]:
four_dim_env = gym.make(FOUR_DIM_ENV_ID)
utils.print_discrete_space(four_dim_env.get_wrapper_attr("discrete_space"))
four_dim_mc = algs.mc.off_policy(four_dim_env)
four_dim_mc.gamma = 0.9
four_dim_mc.behavior = "uniform"
four_dim_output = four_dim_mc.control(
    four_dim_env,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=True,
)

Cart Position: [-2.4  -1.92 -1.44 -0.96 -0.48  0.    0.48  0.96  1.44  1.92  2.4 ]

Cart Velocity: [-2.5 -2.  -1.5 -1.  -0.5  0.   0.5  1.   1.5  2.   2.5]

Pole Angle: [-0.2095 -0.1676 -0.1257 -0.0838 -0.0419  0.      0.0419  0.0838  0.1257
  0.1676  0.2095]

Pole Angular Velocity: [-3.5 -2.8 -2.1 -1.4 -0.7  0.   0.7  1.4  2.1  2.8  3.5]

Failure to meet goal after 100000 iterations.


## Experiment 2

The first experiment failed to converge to an optimal target policy. We realized that the range of starting values enforced by the environment is very small (from -0.05 to 0.05). This meant that for each of our variables, the starting value could only belong to one of the bins. While this is fine theoretically (we do not need the Exploring Starts assumption for Off-Policy MC), it menas that very few of the states are being sampled frequently to make learning practical (theoretically we need to be able to guarantee each state-action pair is visited infinitely). Given our current discretization this would take an infeasibly long time unless we significantly increase the number of bins (which quickly becomes computationally intractable without distributed architecture).

We then remembered that there are only two possible actions: to push left or right. There is no way for the agent to decide how hard to push at any given timestep; it can only apply a pre-determined constant amount of pressure regardless of any state variables. The only thing the agent needs to decide is the direction to move the cart to. Therefore, the cart velocity and angular velocity are practically useless information! The only pertinent information for deciding on a direction to push are the position of the cart and angle of the pole (regardless of velocity).

We will now only consider cart position and pole angle and use $100$ bins for each. Therefore the size of our state space $|S| = 100 \times 100 = 10000$ remains the same. By eliminating the useless velocity variables, we have gained a drastically finer discretization without an increase in the size of the state space!

In [8]:
# Experiment 2: two-dimensional CartPole discretization with the same MC algorithm.


In [9]:
two_dim_env = gym.make(TWO_DIM_ENV_ID)
utils.print_discrete_space(two_dim_env.get_wrapper_attr("discrete_space"))
two_dim_mc = algs.mc.off_policy(two_dim_env)
two_dim_mc.gamma = 0.9
two_dim_mc.behavior = "uniform"
two_dim_output = two_dim_mc.control(
    two_dim_env,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=True,
)

Cart Position: [-2.4   -2.352 -2.304 -2.256 -2.208 -2.16  -2.112 -2.064 -2.016 -1.968
 -1.92  -1.872 -1.824 -1.776 -1.728 -1.68  -1.632 -1.584 -1.536 -1.488
 -1.44  -1.392 -1.344 -1.296 -1.248 -1.2   -1.152 -1.104 -1.056 -1.008
 -0.96  -0.912 -0.864 -0.816 -0.768 -0.72  -0.672 -0.624 -0.576 -0.528
 -0.48  -0.432 -0.384 -0.336 -0.288 -0.24  -0.192 -0.144 -0.096 -0.048
  0.     0.048  0.096  0.144  0.192  0.24   0.288  0.336  0.384  0.432
  0.48   0.528  0.576  0.624  0.672  0.72   0.768  0.816  0.864  0.912
  0.96   1.008  1.056  1.104  1.152  1.2    1.248  1.296  1.344  1.392
  1.44   1.488  1.536  1.584  1.632  1.68   1.728  1.776  1.824  1.872
  1.92   1.968  2.016  2.064  2.112  2.16   2.208  2.256  2.304  2.352
  2.4  ]

Cart Velocity: [-0.2095  -0.20531 -0.20112 -0.19693 -0.19274 -0.18855 -0.18436 -0.18017
 -0.17598 -0.17179 -0.1676  -0.16341 -0.15922 -0.15503 -0.15084 -0.14665
 -0.14246 -0.13827 -0.13408 -0.12989 -0.1257  -0.12151 -0.11732 -0.11313
 -0.10894 -0.10475 -0.10056 -0.

## Experiment 3

The second experiment also failed to converge despite our significantly finer discretization. We came up with the hypothesis that there is too much noise in our behaviour policy; by choosing either action with equal probability this policy is ignoring new information about the state-action pairs which is used to update the target policy. So then our final idea is to bring the behaviour policy closer to the target policy whilst retaining its "softness". This is done by providing a small epsilon term to the algorithm. At any point in an episode, the behaviour policy will choose the current best action (greedy argmax from target policy) with probability $1-\epsilon$, and with probability $\epsilon$ it will pick the non-greedy action for exploration purposes. Thus our behaviour policy is now an $\epsilon$-soft policy, but it still maintains coverage of the target policy.

To do this, however, we need to keep track of the probability for each action chosen by the behaviour policy. This was not needed before because both actions had an equal probability so we could simply use a hard code value of $0.5$. Clearly, $\epsilon \neq 1 - \epsilon \neq 0.5$, so the MC agent records the behaviour-policy probability with each generated transition. We also decided to decrease our gamma value (thereby decreasing the long term return and ultimately punishing the agent for shorter episodes).

In [10]:
# Experiment 3 reuses the registered two-dimensional environment ID and changes only the behaviour policy.

This experiment creates a fresh environment from the same two-dimensional discretized CartPole registration and changes only the behaviour policy configured on `OffPolicyMCAgent`.

In [11]:
# Epsilon-soft behaviour is configured on the package MC control agent.


In [12]:
epsilon_soft_env = gym.make(TWO_DIM_ENV_ID)
utils.print_discrete_space(epsilon_soft_env.get_wrapper_attr("discrete_space"))
epsilon_soft_mc = algs.mc.off_policy(epsilon_soft_env)
epsilon_soft_mc.gamma = 0.3
epsilon_soft_mc.behavior = "epsilon_soft"
epsilon_soft_mc.epsilon = 0.1
epsilon_soft_output = epsilon_soft_mc.control(
    epsilon_soft_env,
    num_timesteps_goal=NUM_TIMESTEPS_GOAL,
    close_env=True,
)

Cart Position: [-2.4   -2.352 -2.304 -2.256 -2.208 -2.16  -2.112 -2.064 -2.016 -1.968
 -1.92  -1.872 -1.824 -1.776 -1.728 -1.68  -1.632 -1.584 -1.536 -1.488
 -1.44  -1.392 -1.344 -1.296 -1.248 -1.2   -1.152 -1.104 -1.056 -1.008
 -0.96  -0.912 -0.864 -0.816 -0.768 -0.72  -0.672 -0.624 -0.576 -0.528
 -0.48  -0.432 -0.384 -0.336 -0.288 -0.24  -0.192 -0.144 -0.096 -0.048
  0.     0.048  0.096  0.144  0.192  0.24   0.288  0.336  0.384  0.432
  0.48   0.528  0.576  0.624  0.672  0.72   0.768  0.816  0.864  0.912
  0.96   1.008  1.056  1.104  1.152  1.2    1.248  1.296  1.344  1.392
  1.44   1.488  1.536  1.584  1.632  1.68   1.728  1.776  1.824  1.872
  1.92   1.968  2.016  2.064  2.112  2.16   2.208  2.256  2.304  2.352
  2.4  ]

Cart Velocity: [-0.2095  -0.20531 -0.20112 -0.19693 -0.19274 -0.18855 -0.18436 -0.18017
 -0.17598 -0.17179 -0.1676  -0.16341 -0.15922 -0.15503 -0.15084 -0.14665
 -0.14246 -0.13827 -0.13408 -0.12989 -0.1257  -0.12151 -0.11732 -0.11313
 -0.10894 -0.10475 -0.10056 -0.

## Limitations

So why did our algorithms fail to converge to a deterministic optimal policy? Perhaps an even finer discretization (larger state space) is needed. perhaps a more selective discretization that involves non-linear transformations of the current intervals are needed (to emphasize states that are less likely to be encountered).

It is also important to note that this is a very limited environment; as mentioned earlier, the only actions the agent can take is to decide the direction in which to push the cart. The agent cannot specify the amount of force to apply at any timestep, nor can it decide not to interfere. It seems nearly impossible that a constant force applied at each timestep could ever enable control of this problem. As the documentation notes, the "center of gravity of the pole varies the amount of energy needed to move the cart underneath it". Perhaps this problem with the given state and action space could be better solved by non-tabular RL methods. Our conclusion is that for any commercial PC and GPU, solving this given problem (keeping the pole balanced indefinitely) with this algorithm and state-action space, is not possible.

# Appendix A

Here are the logs from the four-dimensional discretization target policy tests.

In [13]:
pprint.pprint(four_dim_output)


['Iteration 1000, Test 1: target policy episode length 127',
 'Iteration 2000, Test 2: target policy episode length 91',
 'Iteration 3000, Test 3: target policy episode length 34',
 'Iteration 4000, Test 4: target policy episode length 145',
 'Iteration 5000, Test 5: target policy episode length 78',
 'Iteration 6000, Test 6: target policy episode length 52',
 'Iteration 7000, Test 7: target policy episode length 576',
 'Iteration 8000, Test 8: target policy episode length 128',
 'Iteration 9000, Test 9: target policy episode length 141',
 'Iteration 10000, Test 10: target policy episode length 94',
 'Iteration 11000, Test 11: target policy episode length 115',
 'Iteration 12000, Test 12: target policy episode length 293',
 'Iteration 13000, Test 13: target policy episode length 116',
 'Iteration 14000, Test 14: target policy episode length 84',
 'Iteration 15000, Test 15: target policy episode length 45',
 'Iteration 16000, Test 16: target policy episode length 127',
 'Iteration 17000

# Appendix B

Here are the logs from the two-dimensional discretization target policy tests:

In [14]:
pprint.pprint(two_dim_output)


['Iteration 1000, Test 1: target policy episode length 12',
 'Iteration 2000, Test 2: target policy episode length 31',
 'Iteration 3000, Test 3: target policy episode length 31',
 'Iteration 4000, Test 4: target policy episode length 37',
 'Iteration 5000, Test 5: target policy episode length 22',
 'Iteration 6000, Test 6: target policy episode length 20',
 'Iteration 7000, Test 7: target policy episode length 11',
 'Iteration 8000, Test 8: target policy episode length 44',
 'Iteration 9000, Test 9: target policy episode length 21',
 'Iteration 10000, Test 10: target policy episode length 27',
 'Iteration 11000, Test 11: target policy episode length 15',
 'Iteration 12000, Test 12: target policy episode length 20',
 'Iteration 13000, Test 13: target policy episode length 11',
 'Iteration 14000, Test 14: target policy episode length 22',
 'Iteration 15000, Test 15: target policy episode length 24',
 'Iteration 16000, Test 16: target policy episode length 16',
 'Iteration 17000, Test 17

# Appendix C

Here are the logs from the epsilon-soft behaviour-policy target policy tests:

In [15]:
pprint.pprint(epsilon_soft_output)


['Iteration 1000, Test 1: target policy episode length 36',
 'Iteration 2000, Test 2: target policy episode length 46',
 'Iteration 3000, Test 3: target policy episode length 32',
 'Iteration 4000, Test 4: target policy episode length 87',
 'Iteration 5000, Test 5: target policy episode length 24',
 'Iteration 6000, Test 6: target policy episode length 50',
 'Iteration 7000, Test 7: target policy episode length 25',
 'Iteration 8000, Test 8: target policy episode length 45',
 'Iteration 9000, Test 9: target policy episode length 41',
 'Iteration 10000, Test 10: target policy episode length 59',
 'Iteration 11000, Test 11: target policy episode length 38',
 'Iteration 12000, Test 12: target policy episode length 36',
 'Iteration 13000, Test 13: target policy episode length 58',
 'Iteration 14000, Test 14: target policy episode length 66',
 'Iteration 15000, Test 15: target policy episode length 20',
 'Iteration 16000, Test 16: target policy episode length 40',
 'Iteration 17000, Test 17